In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [2]:
import numpy as np
from loaders._load_vn30_binary import preprocess, VN30, TARGETS
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

In [3]:
X_train, y_train = preprocess("ACB", verbose=True)["train"]

=== Preprocessing ACB ===
Train: (1213, 124) | Valid: None | Test: (328, 124)
Label dist train: Counter({0: 643, 1: 570}), test: Counter({0: 175, 1: 153})


In [4]:
acc = []
for symbol in VN30:
    data = preprocess(symbol, lag=30)
    X_train, Y_train = data["train"]
    X_test, Y_test = data["test"]

    tscv = TimeSeriesSplit(n_splits=5)

    param_dist = {
        "n_estimators": [10, 20, 50, 100],
        "max_depth": [3, 5, 7, 9],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    }

    search = RandomizedSearchCV(
        estimator=RandomForestClassifier(),
        param_distributions=param_dist,
        n_iter=10,        
        cv=tscv,             
        n_jobs=-1,
        random_state=42
    )

    search.fit(X_train, Y_train)

    train_preds = search.predict(X_train)
    test_preds = search.predict(X_test)

    print(f"Symbol: {symbol}")
    print(f"Train Balanced Accuracy: {balanced_accuracy_score(Y_train, train_preds)}")
    print(f"Test Balanced Accuracy: {balanced_accuracy_score(Y_test, test_preds)}")
    print(f"Test Confusion Matrix:\n{confusion_matrix(Y_test, test_preds)}\n")

    acc.append(balanced_accuracy_score(Y_test, test_preds))

# Mean of best 10
mean_acc = np.mean(sorted(acc)[-10:])
print(f"Mean Test Balanced Accuracy (Best 10): {mean_acc}")

Symbol: ACB
Train Balanced Accuracy: 0.7486589724700554
Test Balanced Accuracy: 0.5111858076563959
Test Confusion Matrix:
[[108  67]
 [ 91  62]]

Symbol: BCM
Train Balanced Accuracy: 0.6250600240096038
Test Balanced Accuracy: 0.49186483103879847
Test Confusion Matrix:
[[176  11]
 [135   6]]

Symbol: BID
Train Balanced Accuracy: 0.6699847481827622
Test Balanced Accuracy: 0.51230557572515
Test Confusion Matrix:
[[172  19]
 [120  17]]

Symbol: BVH
Train Balanced Accuracy: 0.8826714801444043
Test Balanced Accuracy: 0.503005103005103
Test Confusion Matrix:
[[168  17]
 [129  14]]

Symbol: CTG
Train Balanced Accuracy: 0.8002546076690424
Test Balanced Accuracy: 0.48479427549195
Test Confusion Matrix:
[[105  51]
 [121  51]]

Symbol: FPT
Train Balanced Accuracy: 0.8124514563106796
Test Balanced Accuracy: 0.5258084924267798
Test Confusion Matrix:
[[91 68]
 [88 81]]

Symbol: GAS
Train Balanced Accuracy: 0.8054531886648083
Test Balanced Accuracy: 0.48242724842097107
Test Confusion Matrix:
[[160  37